<a href="https://colab.research.google.com/github/divyapandya01/table3_mechanisticinterpretability/blob/main/table3qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print([f for f in os.listdir('.') if f.endswith('.py')])
!pip install -q transformers accelerate tqdm scikit-learn matplotlib

['table3_extract_v2.py', 'table3_analyze.py']


In [ ]:
!python table3_extract_v2.py --model Qwen/Qwen2.5-1.5B --smoke

20 concepts, 2400 prompts (4 families x 5 x 24 queries x 5 shot counts)

--- sample prompt ---
tzirwzt -> blicket
ebcwrvm ->

row: {'concept_idx': 0, 'concept_id': 'single_my_nb', 'family': 'single', 'k': 1, 'test_idx': 0, 'query': 'ebcwrvm', 'label': 0}

label balance: [1200 1200]
families: {np.str_('conjunctive'): np.int64(600), np.str_('disjunctive'): np.int64(600), np.str_('negation'): np.int64(600), np.str_('single'): np.int64(600)}


In [ ]:
!python table3_extract_v2.py --model Qwen/Qwen2.5-1.5B

20 concepts, 2400 prompts (4 families x 5 x 24 queries x 5 shot counts)
loading Qwen/Qwen2.5-1.5B on cuda
config.json: 100% 684/684 [00:00<00:00, 2.77MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 10.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 116MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 122MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 141MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   3% 106M/3.09G [00:01<00:30, 98.0MB/s, 6.70MB/s  ] 
model.safetensors: downloading bytes:  12% 363M/3.09G [00:02<00:11, 247MB/s, 31.9MB/s  ]
model.safetensors: downloading bytes:  13% 407M/3.09G [00:03<00:13, 200MB/s, 34.4MB/s  ]
model.safetensors: downloading bytes:  23% 704M/3.09G [00:04<00:08, 286MB/s, 57.4MB/s  ]
model.safetensors: downloading bytes:  26% 811M/3.09G [00:04<00:10, 208MB/s, 64.3MB/s  ]
model.safetensors: downloading bytes:  33% 1.03G/3.09G [00:05<00:11, 183MB/s, 78.3MB/s  ]
model.safetensors

In [ ]:
!python table3_analyze.py --model-name Qwen2.5-1.5B

2400 rows, 29 layers, hidden 1536, 20 concepts, label balance [1200 1200]

behavioral accuracy: 51.7% (chance 50%)
   1-shot:  50.0%
   2-shot:  52.1%
   4-shot:  46.7%
   8-shot:  50.0%
  16-shot:  59.6%
  conjunctive :  53.5%
  disjunctive :  47.7%
  negation    :  54.3%
  single      :  51.2%

  [!] Behavior at/near chance. A strong probe here would mean
      the rule is encoded but NOT USED -- a different claim
      than H2 makes. Flag before writing this up.

[transfer] train 1800 / test 600 (5 held-out concepts)
transfer: 100% 29/29 [00:36<00:00,  1.26s/it]
null: 100% 20/20 [04:08<00:00, 12.44s/it]
  null 50.2% +/- 0.9  thresh 60.2%
  peak 66.2% @ L15  L_E=5  L_S=None

[item] train 1100 / test 1300
item: 100% 29/29 [00:20<00:00,  1.39it/s]
null: 100% 20/20 [02:42<00:00,  8.14s/it]
  null 50.0% +/- 0.7
  peak 65.8% @ L17  L_E=3  L_S=7

[by shot count, transfer split]
k=1: 100% 29/29 [00:11<00:00,  2.48it/s]
  k= 1: peak  57.5% @ L10
k=2: 100% 29/29 [00:11<00:00,  2.54it/s]
  k= 

In [ ]:
from google.colab import files
files.download("activations.npy")
files.download("rows.json")
files.download("table3_row.json")
files.download("table3_curve.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

acts = np.load("activations.npy")
rows = json.loads(open("rows.json").read())
y   = np.array([r["label"] for r in rows])
fam = np.array([r["family"] for r in rows])

def probe(L, labels, tr, te):
    X = acts[:, L, :].astype(np.float32)
    sc = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=2000, C=0.1).fit(sc.transform(X[tr]), labels[tr])
    return clf.score(sc.transform(X[te]), labels[te])

print("FAMILY HOLDOUT — train on 3 families, test on the 4th\n")
peaks = {}
for held in sorted(set(fam)):
    te, tr = fam == held, fam != held
    a = np.array([probe(L, y, tr, te) for L in range(acts.shape[1])])
    peaks[held] = a.max()
    print(f"  held out {held:12s}: peak {100*a.max():5.1f}% @ L{int(a.argmax())}")

# null on one fold
rng = np.random.RandomState(0)
held = sorted(set(fam))[0]
te, tr = fam == held, fam != held
null = np.mean([np.mean([probe(L, rng.permutation(y), tr, te)
                         for L in range(0, acts.shape[1], 6)]) for _ in range(10)])
print(f"\n  permutation null: {100*null:.1f}%")
print(f"  mean peak across folds: {100*np.mean(list(peaks.values())):.1f}%")

FAMILY HOLDOUT — train on 3 families, test on the 4th

  held out conjunctive : peak  69.8% @ L15
  held out disjunctive : peak  63.3% @ L23
  held out negation    : peak  66.2% @ L15
  held out single      : peak  67.8% @ L21

  permutation null: 49.5%
  mean peak across folds: 66.8%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/table3_qwen_2026-08-30"
!cp activations.npy rows.json table3_row.json table3_curve.png *.py "/content/drive/MyDrive/table3_qwen_2026-08-30/"

Mounted at /content/drive
cp: cannot stat 'activations.npy': No such file or directory
cp: cannot stat 'rows.json': No such file or directory
cp: cannot stat 'table3_row.json': No such file or directory
cp: cannot stat 'table3_curve.png': No such file or directory
cp: cannot stat '*.py': No such file or directory
